# Multimodal Animal Verification Pipeline — Demo

| Component | Model | Task |
|-----------|-------|------|
| NLP | `dslim/bert-base-NER` (fine-tuned) | Extract `ANIMAL` entities from free-form text |
| CV  | `ResNet-18` (fine-tuned) | Classify the animal present in an image |

---

## 0. Setup

In [ ]:
import os
import sys
import glob
import warnings

import torch
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, HTML

# Add task_2/ root so that 'src' is resolvable as a package
ROOT = os.path.abspath('..')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

warnings.filterwarnings('ignore')

from src.classifier.inference import load_model as load_cv_model
from src.classifier.inference import predict as predict_cv
from src.ner.inference import load_ner_pipeline, extract_animal_names

print(f"Python: {sys.executable}")

In [ ]:
CV_MODEL_PATH  = "../models/best_animal_classifier.pth"
NER_MODEL_PATH = "../models/ner_animal/best_model"
DATA_DIR       = "../data/raw-img"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

print("Loading CV model...")
cv_model, class_names = load_cv_model(CV_MODEL_PATH, device=DEVICE)

print("Loading NER model...")
ner_pipeline = load_ner_pipeline(
    NER_MODEL_PATH,
    device=0 if DEVICE.type == "cuda" else -1,
)

print(f"\nReady. Classes: {class_names}")

In [ ]:
def get_first_image(animal: str) -> str:
    """Return the path to the first available image for a given animal class."""
    pattern = os.path.join(DATA_DIR, animal, "*")
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No images found for '{animal}' in {DATA_DIR}/{animal}/")
    return files[0]

# Verify dataset is accessible
for cls in class_names:
    path = get_first_image(cls)
    print(f"  {cls:<12} -> {os.path.basename(path)}")

---
## Visualisation Helper

In [ ]:
def evaluate_and_display(
    text: str,
    image_path: str,
    min_confidence: float = 0.0,
) -> bool:
    """
    Run the full multimodal pipeline and render a structured result card.

    Args:
        text:           Free-form input text.
        image_path:     Path to the image to classify.
        min_confidence: Minimum NER entity confidence threshold.

    Returns:
        Boolean match result.
    """
    # 1. NER inference
    extracted = extract_animal_names(text, ner_pipeline, min_confidence)
    extracted_lower = [a.lower() for a in extracted]

    # 2. CV inference
    cv_results = predict_cv(image_path, cv_model, class_names, DEVICE, top_k=3)
    top_animal, top_conf = cv_results[0]

    # 3. Match
    is_match = top_animal.lower() in extracted_lower

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(10, 4),
                             gridspec_kw={"width_ratios": [1, 1.4]})

    ax_img = axes[0]
    try:
        img = Image.open(image_path).convert("RGB")
        ax_img.imshow(img)
    except (FileNotFoundError, OSError):
        ax_img.text(0.5, 0.5, "Image not found",
                    ha="center", va="center", fontsize=12, color="gray")
    ax_img.axis("off")
    ax_img.set_title("Input Image", fontsize=11, fontweight="bold", pad=8)

    ax_bar = axes[1]
    labels = [r[0].capitalize() for r in cv_results]
    values = [r[1] for r in cv_results]
    colors = [
        "#2ecc71" if is_match and i == 0 else
        "#e74c3c" if not is_match and i == 0 else
        "#95a5a6"
        for i in range(len(labels))
    ]
    bars = ax_bar.barh(labels[::-1], values[::-1], color=colors[::-1],
                       height=0.5, edgecolor="white")
    ax_bar.set_xlim(0, 100)
    ax_bar.set_xlabel("Confidence (%)", fontsize=10)
    ax_bar.set_title("CV Top-3 Predictions", fontsize=11, fontweight="bold", pad=8)
    ax_bar.spines[["top", "right"]].set_visible(False)
    for bar, val in zip(bars, values[::-1]):
        ax_bar.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                    f"{val:.1f}%", va="center", fontsize=9)

    plt.tight_layout()
    plt.show()

    # HTML result card
    match_color = "#27ae60" if is_match else "#c0392b"
    match_label = "MATCH - TRUE" if is_match else "MISMATCH - FALSE"
    ner_display = ", ".join(f"<code>{a}</code>" for a in extracted) if extracted else "<i>none detected</i>"

    html = f"""
    <div style="
        font-family: 'Segoe UI', Arial, sans-serif;
        border: 1px solid #e0e0e0;
        border-left: 5px solid {match_color};
        border-radius: 6px;
        padding: 16px 20px;
        max-width: 680px;
        background: #fafafa;
        line-height: 1.7;
    ">
        <p style="margin:0 0 8px;"><b>Input text:</b> <i>"{text}"</i></p>
        <hr style="border:none; border-top:1px solid #e8e8e8; margin:8px 0;">
        <p style="margin:4px 0;"><b>NER extraction:</b> {ner_display}</p>
        <p style="margin:4px 0;"><b>CV prediction:</b>
            <code>{top_animal.capitalize()}</code>
            <span style="color:#888; font-size:0.9em;">({top_conf:.1f}%)</span>
        </p>
        <hr style="border:none; border-top:1px solid #e8e8e8; margin:10px 0 8px;">
        <p style="margin:0; font-size:1.05em;">
            <b>Result:</b>
            <span style="color:{match_color}; font-weight:700; font-size:1.1em;">
                {match_label}
            </span>
        </p>
    </div>
    """
    display(HTML(html))
    return is_match

---
## 1. Standard Use Cases

Baseline tests where the text is straightforward and either clearly matches or contradicts the image.

In [ ]:
# Test 1.1 - Perfect match
# Expected: TRUE
image_cow = get_first_image("cow")

evaluate_and_display(
    text="There is a large brown cow grazing in the picture.",
    image_path=image_cow,
)

In [ ]:
# Test 1.2 - Clear mismatch
# Expected: FALSE - image contains a cow, text mentions a cat
evaluate_and_display(
    text="Look at this cute little cat sleeping on the sofa.",
    image_path=image_cow,
)

---
## 2. Edge Cases

Tests that probe robustness: ambiguous phrasing, multiple entities, zero entities, and capitalisation variation.

In [ ]:
# Edge case 2.1 - Multiple animals in text (logical OR)
# Image is a cow. Text mentions both horse and cow.
# Expected: TRUE - match on 'cow'
evaluate_and_display(
    text="I can't tell if this is a horse or a cow, but it's enormous.",
    image_path=image_cow,
)

In [ ]:
# Edge case 2.2 - Zero entities in text (false-positive prevention)
# Image contains an animal, but text describes only the background.
# Expected: FALSE - no animal entity extracted, no match possible
evaluate_and_display(
    text="This is a beautiful green landscape with a clear blue sky.",
    image_path=image_cow,
)

In [ ]:
# Edge case 2.3 - Capitalisation and punctuation variation
# NER and CV comparison must be case-insensitive.
# Expected: TRUE
evaluate_and_display(
    text="COW!! There's definitely a COW in this photo.",
    image_path=image_cow,
)

In [ ]:
# Edge case 2.4 - Indirect / figurative reference
# The text does not name the animal directly.
# Expected: FALSE - NER cannot extract an implicit reference
evaluate_and_display(
    text="The animal in the image gives a lot of milk and says moo.",
    image_path=image_cow,
)

In [ ]:
# Edge case 2.5 - NER confidence threshold
# Raising min_confidence filters out low-certainty extractions.
# Expected: depends on model confidence for 'cow-like'
evaluate_and_display(
    text="There might be some kind of cow-like creature here.",
    image_path=image_cow,
    min_confidence=0.99,
)

---
## 3. Cross-class Verification

End-to-end tests across several supported animal classes to confirm the pipeline generalises beyond a single class.

In [ ]:
cross_class_tests = [
    ("A dog is running in the park.",             "dog"),
    ("There is a large elephant in the savanna.", "elephant"),
    ("A butterfly rests on a flower.",            "butterfly"),
    ("A spider is weaving its web.",              "spider"),
]

for text, animal in cross_class_tests:
    path = get_first_image(animal)
    print(f"\nTest: {text}")
    evaluate_and_display(text=text, image_path=path)